In [ ]:
# imports
%load_ext autoreload
%autoreload 2
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
# ad-hoc fix for imports 
import os
import sys
dirname = os.getcwd()
sys.path.insert(0, os.path.join(dirname, '../..'))
import tdc_sampler
from Utils.inference_utils import median_sigma_from_samples
from Utils.mcmc_utils import median_and_uncertainty, DE_fom

## Step 1: alpaca --> fasttdc ##

1. Store ground truths in correct format
2. Create .h5 file with the fpd_samps, lens_params_samps

In [ ]:
# Understand samples file:
my_file = 'DataVectors/alpaca/samples_JWST_doubles.h5'
with h5py.File(my_file, 'r') as f:
    # List all keys at the root level
    print("Keys in the HDF5 file:")
    print(list(f.keys()))

Ordering of parameters: 

LENS_PARAM_NAMES = [
    "lens_theta_E",
    "lens_gamma",
    "lens_e1",
    "lens_e2",
    "lens_center_x",
    "lens_center_y",
    "lens_gamma1",
    "lens_gamma2",
]

LENS_LIGHT_PARAM_NAMES = [
    "light_Re_L",
    "light_n_L",
    "light_e1_L",
    "light_e2_L",
    "log_light_amp_L",
]

In [ ]:
# Understand truth file:
my_file = 'DataVectors/alpaca/truth_JWST_doubles.h5'
with h5py.File(my_file, 'r') as f:
    # List all keys at the root level
    print("Keys in the HDF5 file:")
    print(list(f.keys()))

In [ ]:
# FUNCTION to convert alpaca samples/truths into a fasttdc data_vector_dict
from Utils.data_vector_utils import emulate_measurements

def dv_dict_from_alpaca(alpaca_samples_h5, alpaca_truth_h5, td_meas_err_percent=0.05):
    """
    Args: 

    Returns: 
        (dict) data_vector_dict with keys: 
            [ 'td_measured':,
              'td_likelihood_prec':,
              'td_likelihood_prefactors':,
              'fpd_samples':,
              'gamma_pred_samples':,
              'z_lens':,
              'z_src': ]
    """

    # read in from samples file
    with h5py.File(alpaca_samples_h5, 'r') as f:
        cidx_samples = f['catalog_idxs'][:]
        fpd_samps = f['fpd_samples'][:]
        lens_param_samps_unordered = f['lens_param_samples'][:]
        # NOTE: HARDCODED - reorder to fasttdc convention
        lens_param_samps = np.asarray([
            lens_param_samps_unordered[:,:,0], # theta_E
            lens_param_samps_unordered[:,:,6], # gamma1
            lens_param_samps_unordered[:,:,7], # gamma2
            lens_param_samps_unordered[:,:,1], # power-law slope
            lens_param_samps_unordered[:,:,2], # e1
            lens_param_samps_unordered[:,:,3], # e2
            lens_param_samps_unordered[:,:,4], # x_lens
            lens_param_samps_unordered[:,:,5]  # y_lens
        ])
        lens_param_samps = lens_param_samps.transpose(1,2,0)

    with h5py.File(alpaca_truth_h5, 'r') as f:
        # set up indexing
        cidx_truth = f['catalog_idxs'][:]
        truth_idxs = np.isin(cidx_truth,cidx_samples)
        # pull truth values for only the lens indices in alpaca_samples (the
        # idea is that you might have only modeled 1/2 of your truth catalog, for example)
        z_src_all = f['z_src'][:]
        z_src = z_src_all[truth_idxs]
        z_lens_all = f['z_lens'][:]
        z_lens = z_lens_all[truth_idxs]
        td_truth_all = f['time_delays_truth'][:]
        td_truth = td_truth_all[truth_idxs]

    # emulate time-delay measurements with % error given by td_meas_err_percent
    td_meas, td_meas_prec = emulate_measurements(td_truth,measurement_error_percent=td_meas_err_percent)

    # add repeats on 2nd axis for compatibility with importance samples
    num_fpd_samps = np.shape(fpd_samps)[1]
    td_meas = np.repeat(td_meas[:, np.newaxis, :],
        num_fpd_samps, axis=1)
    td_meas_prec = np.repeat(td_meas_prec[:, np.newaxis, :, :],
        num_fpd_samps, axis=1)
    
    # add prefactors to track wheter its a 1D vs 3D Gaussian evaluation
    num_td = np.shape(fpd_samps)[2]
    td_gaussian_prefactor = np.log( (1/(2*np.pi)**(num_td/2)) / 
        np.sqrt(np.linalg.det(np.linalg.inv(td_meas_prec))) )

    # create the data vector dictionary
    data_vector_dict = {   
        'td_measured':td_meas,
        'td_likelihood_prec':td_meas_prec,
        'td_likelihood_prefactors':td_gaussian_prefactor,
        'fpd_samples':fpd_samps,
        'lens_param_samples':lens_param_samps,
        'z_lens':z_lens,
        'z_src':z_src,
    }

    return data_vector_dict

In [ ]:
# convert with the new function
data_vector_dict_dbls = dv_dict_from_alpaca('DataVectors/alpaca/samples_JWST_doubles.h5',
    'DataVectors/alpaca/truth_JWST_doubles.h5')

In [ ]:
data_vector_dict_quads = dv_dict_from_alpaca('DataVectors/alpaca/samples_JWST_quads.h5',
    'DataVectors/alpaca/truth_JWST_quads.h5')

In [ ]:
# set up a likelihood object
lklhd_obj = tdc_sampler.TDCLikelihood(
    fpd_sample_shape=np.shape(data_vector_dict_dbls['fpd_samples']), # pre-define input shape for fermat potential differences
    cosmo_model='LCDM', # infers four params: [H0,OmegaM,mu(gamma),sigma(gamma)]
    use_astropy=True, # relic option, we always use astropy right now
    use_gamma_info=True) # whether to infer a population over gamma_lens or not

my_chain_dbls = tdc_sampler.fast_TDC([lklhd_obj], [data_vector_dict_dbls], num_emcee_samps=1000,
    n_walkers=20, use_mpi=False, use_multiprocess=False, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
# set up a likelihood object
lklhd_obj_quads = tdc_sampler.TDCLikelihood(
    fpd_sample_shape=np.shape(data_vector_dict_quads['fpd_samples']), # pre-define input shape for fermat potential differences
    cosmo_model='LCDM', # infers four params: [H0,OmegaM,mu(gamma),sigma(gamma)]
    use_astropy=True, # relic option, we always use astropy right now
    use_gamma_info=True) # whether to infer a population over gamma_lens or not

my_chain_quads = tdc_sampler.fast_TDC([lklhd_obj_quads], [data_vector_dict_quads], num_emcee_samps=1000,
    n_walkers=20, use_mpi=False, use_multiprocess=False, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
my_chain_both = tdc_sampler.fast_TDC([lklhd_obj,lklhd_obj_quads], 
    [data_vector_dict_dbls,data_vector_dict_quads], num_emcee_samps=1000,
    n_walkers=20, use_mpi=False, use_multiprocess=False, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
# Verify things ran correctly by looking at the chains

def plot_convergence_by_walker(samples_mcmc, param_mcmc, verbose = False):
    n_params = samples_mcmc.shape[2]
    n_step = int(samples_mcmc.shape[1])
    chain = samples_mcmc
    mean_pos = np.zeros((n_params, n_step))
    median_pos = np.zeros((n_params, n_step))
    std_pos = np.zeros((n_params, n_step))
    q16_pos = np.zeros((n_params, n_step))
    q84_pos = np.zeros((n_params, n_step))
    # chain = np.empty((nwalker, nstep, ndim), dtype = np.double)
    for i in np.arange(n_params):
        for j in np.arange(n_step):
            mean_pos[i][j] = np.mean(chain[:, j, i])
            median_pos[i][j] = np.median(chain[:, j, i])
            std_pos[i][j] = np.std(chain[:, j, i])
            q16_pos[i][j] = np.percentile(chain[:, j, i], 16.)
            q84_pos[i][j] = np.percentile(chain[:, j, i], 84.)
    fig, ax = plt.subplots(n_params, sharex=True, figsize=(16, 2 * n_params))
    if n_params == 1: ax = [ax]
    last = n_step
    burnin = int((9.*n_step) / 10.) #get the final value on the last 10% on the chain
    for i in range(n_params):
        if verbose :
            print(param_mcmc[i], '{:.4f} +/- {:.4f}'.format(median_pos[i][last - 1], (q84_pos[i][last - 1] - q16_pos[i][last - 1]) / 2))
        ax[i].plot(median_pos[i][:last], c='g')
        ax[i].axhline(np.median(median_pos[i][burnin:last]), c='r', lw=1)
        ax[i].fill_between(np.arange(last), q84_pos[i][:last], q16_pos[i][:last], alpha=0.4)
        ax[i].set_ylabel(param_mcmc[i], fontsize=10)
        ax[i].set_xlim(0, last)
    return fig

#with h5py.File('DataVectors/gold/baseline_chain.h5','r') as h5:
#    test_chain = h5['mcmc_chain'][:]
plot_convergence_by_walker(np.transpose(my_chain_both,axes=(1,0,2)),
    ['$H_0$','$\Omega_M$',
     r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$'])

In [ ]:
# Corner plots!

H0_TRUTH = 70.
OMEGAM_TRUTH = 0.27
MEAN_GAMMA_TRUTH = 2.0
SIGMA_GAMMA_TRUTH = 0.1

from scipy.stats import norm
import corner

exp_chains = [
    np.transpose(my_chain_dbls,axes=(1,0,2)),
    np.transpose(my_chain_quads,axes=(1,0,2)),
    np.transpose(my_chain_both,axes=(1,0,2))]
exp_names = ['2 Doubles',
             '8 Quads',
             '10 Combined']

num_chains = len(exp_chains)
burnin = [200,200,200]
cmap = plt.get_cmap('ocean')
colors = ['C0','C1','C2','C3','C4'] #goldenrod',
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    median_and_uncertainty(exp_chain,burnin[i])
    #zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$\Omega_M$',
                r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$'],
            dpi=300,#truths=[H0_TRUTH,OMEGAM_TRUTH,MEAN_GAMMA_TRUTH,SIGMA_GAMMA_TRUTH],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':40},smooth=2)

    else:

        corner.corner(exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_M$',
                r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$'],
            dpi=300,#truths=[H0_TRUTH,OMEGAM_TRUTH,MEAN_GAMMA_TRUTH,SIGMA_GAMMA_TRUTH],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':40},smooth=2)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=10))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.1f$\pm$%.2f \n $\Omega_m$=%.3f$\pm$%.3f'%(
        np.round(h0,decimals=2), np.round(h0_sigma,decimals=2), 
        np.round(OmegaM,decimals=3), np.round(OmegaM_sigma,decimals=3)))


axes = np.array(figure.axes).reshape((4, 4))
bounds = [[65,95],[0.05,0.5],[1.8,2.2],[0.,0.2]]
for r in range(0,4):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

axes[0,num_params-1].legend(custom_lines,custom_labels,frameon=False,fontsize=20)
#plt.tight_layout()
#plt.savefig('/Users/smericks/Desktop/gold_vs_silver.pdf',bbox_inches='tight')